<a href="https://colab.research.google.com/github/madan-genai/machine-learning/blob/main/Bidirectional_LSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

In [3]:
df=pd.read_csv("/content/FA-KES-Dataset.csv", encoding='latin1')

In [4]:
df.head()

,unit_id,article_title,article_content,source,date,location,labels
0,1914947530,Syria attack symptoms consistent with nerve ag...,Wed 05 Apr 2017 Syria attack symptoms consiste...,nna,4/5/2017,idlib,0
1,1914947532,Homs governor says U.S. attack caused deaths b...,Fri 07 Apr 2017 at 0914 Homs governor says U.S...,nna,4/7/2017,homs,0
2,1914947533,Death toll from Aleppo bomb attack at least 112,Sun 16 Apr 2017 Death toll from Aleppo bomb at...,nna,4/16/2017,aleppo,0
3,1914947534,Aleppo bomb blast kills six Syrian state TV,Wed 19 Apr 2017 Aleppo bomb blast kills six Sy...,nna,4/19/2017,aleppo,0
4,1914947535,29 Syria Rebels Dead in Fighting for Key Alepp...,Sun 10 Jul 2016 29 Syria Rebels Dead in Fighti...,nna,7/10/2016,aleppo,0


In [6]:
df.isnull().sum()

,0
unit_id,0
article_title,0
article_content,0
source,0
date,0
location,0
labels,0


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 804 entries, 0 to 803
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   unit_id          804 non-null    int64 
 1   article_title    804 non-null    object
 2   article_content  804 non-null    object
 3   source           804 non-null    object
 4   date             804 non-null    object
 5   location         804 non-null    object
 6   labels           804 non-null    int64 
dtypes: int64(2), object(5)
memory usage: 44.1+ KB


In [8]:
df.describe()

,unit_id,labels
count,8.040000e+02,804.000000
mean,1.936024e+09,0.529851
std,1.876968e+07,0.499419
min,1.914948e+09,0.000000
25%,1.923848e+09,0.000000
50%,1.924058e+09,1.000000
75%,1.962496e+09,1.000000
max,1.965511e+09,1.000000


In [9]:
df.shape

(804, 7)

In [10]:
df=df.dropna()

In [11]:
df.head()

,unit_id,article_title,article_content,source,date,location,labels
0,1914947530,Syria attack symptoms consistent with nerve ag...,Wed 05 Apr 2017 Syria attack symptoms consiste...,nna,4/5/2017,idlib,0
1,1914947532,Homs governor says U.S. attack caused deaths b...,Fri 07 Apr 2017 at 0914 Homs governor says U.S...,nna,4/7/2017,homs,0
2,1914947533,Death toll from Aleppo bomb attack at least 112,Sun 16 Apr 2017 Death toll from Aleppo bomb at...,nna,4/16/2017,aleppo,0
3,1914947534,Aleppo bomb blast kills six Syrian state TV,Wed 19 Apr 2017 Aleppo bomb blast kills six Sy...,nna,4/19/2017,aleppo,0
4,1914947535,29 Syria Rebels Dead in Fighting for Key Alepp...,Sun 10 Jul 2016 29 Syria Rebels Dead in Fighti...,nna,7/10/2016,aleppo,0


In [12]:
X=df.drop("labels",axis=1)

In [13]:
y=df["labels"]

In [20]:
import tensorflow as tf

In [21]:
from tensorflow.keras.layers import Embedding
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.text import one_hot
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Bidirectional
from tensorflow.keras.layers import Dropout

In [22]:
vocab_size=5000

In [23]:
messages=X.copy()

In [24]:
messages.head(5)

,unit_id,article_title,article_content,source,date,location
0,1914947530,Syria attack symptoms consistent with nerve ag...,Wed 05 Apr 2017 Syria attack symptoms consiste...,nna,4/5/2017,idlib
1,1914947532,Homs governor says U.S. attack caused deaths b...,Fri 07 Apr 2017 at 0914 Homs governor says U.S...,nna,4/7/2017,homs
2,1914947533,Death toll from Aleppo bomb attack at least 112,Sun 16 Apr 2017 Death toll from Aleppo bomb at...,nna,4/16/2017,aleppo
3,1914947534,Aleppo bomb blast kills six Syrian state TV,Wed 19 Apr 2017 Aleppo bomb blast kills six Sy...,nna,4/19/2017,aleppo
4,1914947535,29 Syria Rebels Dead in Fighting for Key Alepp...,Sun 10 Jul 2016 29 Syria Rebels Dead in Fighti...,nna,7/10/2016,aleppo


In [25]:
import nltk
import re
from nltk.corpus import stopwords

In [26]:
nltk.download("stopwords")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [27]:
messages.reset_index(inplace=True)

In [30]:
from nltk.stem.porter import PorterStemmer
ps=PorterStemmer()
corpus=[]
for i in range(0,len(messages)):
  print(i)
  review = re.sub('[^a-zA-Z]', ' ', messages['article_title'][i])
  review = review.lower()
  review = review.split()

  review = [ps.stem(word) for word in review if not word in stopwords.words("english")]
  review = " ".join(review)
  corpus.append(review)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276
27

In [32]:
corpus

['syria attack symptom consist nerv agent use',
 'hom governor say u attack caus death doesnt see big human loss',
 'death toll aleppo bomb attack least',
 'aleppo bomb blast kill six syrian state tv',
 'syria rebel dead fight key aleppo road',
 'suicid bomb kill least northeast syria',
 'dead heavi u raid syria stronghold',
 'suicid bomber kill assad clan hometown',
 'explos rock town damascu',
 'damascu explos due rocket bomb',
 'syrian regim step aerial assault douma',
 'hizballah lead regim offens southern syria',
 'syrian opposit remain divid',
 'video show murder syrian activist',
 'syria nusra front stage deadli suicid bomb aleppo',
 'regim troop thwart rebel attack syria aleppo',
 'ahrar al sham leader kill syria',
 'barrel bomb kill town syria',
 'rebel advanc north western syria',
 'isra strike syrian town kill pro regim fighter',
 'syria armi plane crash rebel held town',
 'syrian regim reveng attack kill score qalamoun',
 'chemic massacr idlib defi world',
 'bu bomb mark en

In [35]:
one_hot_resp=[one_hot(word,vocab_size)for word in corpus]
one_hot_resp

[[4119, 3271, 2011, 4469, 3310, 1297, 1008],
 [1888, 4023, 867, 2674, 3271, 459, 1043, 4762, 716, 2097, 580, 2438],
 [1043, 1942, 590, 4811, 3271, 1582],
 [590, 4811, 2128, 1823, 1272, 3692, 4667, 1324],
 [4119, 3907, 3913, 1313, 3751, 590, 2898],
 [78, 4811, 1823, 1582, 1632, 4119],
 [3913, 2603, 2674, 4129, 4119, 4221],
 [78, 2979, 1823, 4083, 1826, 1863],
 [1206, 3429, 4463, 3859],
 [3859, 1206, 1662, 4346, 4811],
 [3692, 4110, 229, 4230, 1578, 3455],
 [904, 3275, 4110, 4580, 2533, 4119],
 [3692, 4958, 4136, 3254],
 [1142, 624, 2867, 3692, 4527],
 [4119, 242, 1408, 1640, 3446, 78, 4811, 590],
 [4110, 1424, 4268, 3907, 3271, 4119, 590],
 [2117, 4336, 2307, 3959, 1823, 4119],
 [3053, 4811, 1823, 4463, 4119],
 [3907, 3765, 1231, 3036, 4119],
 [2505, 2462, 3692, 4463, 1823, 1017, 4110, 2205],
 [4119, 3675, 1082, 4763, 3907, 2651, 4463],
 [3692, 4110, 2762, 3271, 1823, 3434, 2514],
 [1522, 723, 3038, 1043, 2882],
 [3377, 4811, 476, 4054, 749, 486, 4694, 4119],
 [1272, 1823, 590, 4811, 48

In [37]:
sent_length=20
embedded_docs=pad_sequences(one_hot_resp,padding='pre',maxlen=sent_length)

embedded_docs


array([[   0,    0,    0, ..., 3310, 1297, 1008],
       [   0,    0,    0, ..., 2097,  580, 2438],
       [   0,    0,    0, ..., 4811, 3271, 1582],
       ...,
       [   0,    0,    0, ..., 3692,  590, 4030],
       [   0,    0,    0, ..., 2236, 1429, 4119],
       [   0,    0,    0, ..., 4652, 3666, 2074]], dtype=int32)

In [47]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Bidirectional, Input

embedding_feature_vectors = 40

model = Sequential()

model.add(Input(shape=(sent_length,)))

model.add(Embedding(input_dim=vocab_size,
                    output_dim=embedding_feature_vectors))

model.add(Bidirectional(LSTM(100)))

model.add(Dense(1, activation="sigmoid"))

model.compile(loss="binary_crossentropy",
              optimizer="adam",
              metrics=["accuracy"])

model.summary()

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_6 (Embedding)         │ (None, 20, 40)         │       200,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_5 (Bidirectional) │ (None, 200)            │       112,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │           201 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 313,001 (1.19 MB)

 Trainable params: 313,001 (1.19 MB)

 Non-trainable params: 0 (0.00 B)

In [48]:
import numpy as np
X_final=np.array(embedded_docs)
y_final=np.array(y)

In [49]:
## train test split

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_final, y_final, test_size=0.33, random_state=42)

In [51]:
## Model Training
model.fit(X_train,y_train,validation_data=(X_test,y_test),epochs=10,batch_size=32)

Epoch 1/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 5s 69ms/step - accuracy: 0.5172 - loss: 0.6940 - val_accuracy: 0.5564 - val_loss: 0.6914
Epoch 2/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 42ms/step - accuracy: 0.5083 - loss: 0.6916 - val_accuracy: 0.5564 - val_loss: 0.6882
Epoch 3/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 40ms/step - accuracy: 0.5136 - loss: 0.6857 - val_accuracy: 0.5376 - val_loss: 0.6901
Epoch 4/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 38ms/step - accuracy: 0.6537 - loss: 0.6579 - val_accuracy: 0.4925 - val_loss: 0.7145
Epoch 5/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 41ms/step - accuracy: 0.7828 - loss: 0.5714 - val_accuracy: 0.4850 - val_loss: 0.8701
Epoch 6/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 39ms/step - accuracy: 0.8176 - loss: 0.4257 - val_accuracy: 0.5038 - val_loss: 0.7812
Epoch 7/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 38ms/step - accuracy: 0.8477 - loss: 0.4119 - val_accuracy: 0.5075 - val_loss: 0.9661
Epoch 8/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 38ms/step - accuracy: 0.8454 - loss: 0.3465 - val_accuracy: 0.5038 - v

In [52]:
y_pred=model.predict(X_test)

9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step


In [53]:
import numpy as np

y_pred=np.where(y_pred>=0.5,1,0)

In [54]:
from sklearn.metrics import confusion_matrix,accuracy_score
confusion_matrix(y_test,y_pred)

array([[50, 68],
       [60, 88]])

In [55]:
print(accuracy_score(y_test,y_pred))

0.518796992481203
